In [29]:
import pandas as pd

## Импорт и считывание файла

In [30]:
df = pd.read_csv(
    'feed-views.log',
    sep='\t',
    names=['datetime', 'user'],  # задаем имена колонок
    engine='python'
)
# Преобразуем колонку 'datetime' в тип datetime
df['datetime'] = pd.to_datetime(df['datetime'])

# Теперь можно извлечь компоненты
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['day'] = df['datetime'].dt.day
df['hour'] = df['datetime'].dt.hour
df['minute'] = df['datetime'].dt.minute
df['second'] = df['datetime'].dt.second

print(df.head())

                    datetime   user  year  month  day  hour  minute  second
0 2020-04-17 12:01:08.463179  artem  2020      4   17    12       1       8
1 2020-04-17 12:01:23.743946  artem  2020      4   17    12       1      23
2 2020-04-17 12:27:30.646665  artem  2020      4   17    12      27      30
3 2020-04-17 12:35:44.884757  artem  2020      4   17    12      35      44
4 2020-04-17 12:35:52.735016  artem  2020      4   17    12      35      52


## Устанавил столбец user как индекс и добавил распределение по временам дня

In [31]:
bins = [0, 4, 7, 11, 17, 20, 24]
labels = ["night", "early morning", "morning", "afternoon", "early evening", "evening"]

# Создаем колонку 'daytime' с помощью cut
df['daytime'] = pd.cut(df['hour'], bins=bins, labels=labels, right=False)

# Назначаем колонку 'user' индексом
df.set_index('user', inplace=True)

In [32]:
df

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
artem,2020-04-17 12:01:08.463179,2020,4,17,12,1,8,afternoon
artem,2020-04-17 12:01:23.743946,2020,4,17,12,1,23,afternoon
artem,2020-04-17 12:27:30.646665,2020,4,17,12,27,30,afternoon
artem,2020-04-17 12:35:44.884757,2020,4,17,12,35,44,afternoon
artem,2020-04-17 12:35:52.735016,2020,4,17,12,35,52,afternoon
...,...,...,...,...,...,...,...,...
valentina,2020-05-21 18:45:20.441142,2020,5,21,18,45,20,early evening
maxim,2020-05-21 23:03:06.457819,2020,5,21,23,3,6,evening
pavel,2020-05-21 23:23:49.995349,2020,5,21,23,23,49,evening


In [33]:
df.head()

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
artem,2020-04-17 12:01:08.463179,2020,4,17,12,1,8,afternoon
artem,2020-04-17 12:01:23.743946,2020,4,17,12,1,23,afternoon
artem,2020-04-17 12:27:30.646665,2020,4,17,12,27,30,afternoon
artem,2020-04-17 12:35:44.884757,2020,4,17,12,35,44,afternoon
artem,2020-04-17 12:35:52.735016,2020,4,17,12,35,52,afternoon


In [34]:
df.tail()

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
valentina,2020-05-21 18:45:20.441142,2020,5,21,18,45,20,early evening
maxim,2020-05-21 23:03:06.457819,2020,5,21,23,3,6,evening
pavel,2020-05-21 23:23:49.995349,2020,5,21,23,23,49,evening
artem,2020-05-21 23:49:22.386789,2020,5,21,23,49,22,evening
artem,2020-05-22 10:36:14.662600,2020,5,22,10,36,14,morning


In [35]:
category_counts = df['daytime'].value_counts()

In [36]:
print(category_counts)

daytime
evening          509
afternoon        252
early evening    145
night            129
morning           36
early morning      5
Name: count, dtype: int64


## Проверки

In [37]:
total_elements = df.count().sum()

In [38]:
print(total_elements)

8608


In [39]:
df_sorted = df.sort_values(by = ["hour", "minute", "second"])

In [40]:
df_sorted.head()
df = df_sorted

## Разбиение + математические вычисления

In [41]:
# Для утра - минимальный час
morning_rows = df[df['daytime'] == 'morning']

if not morning_rows.empty:
    min_hour_morning = morning_rows['hour'].min()

    visitors_at_min_hour = df[(df['daytime'] == 'morning') & (df['hour'] == min_hour_morning)]

    if not visitors_at_min_hour.empty:
        user_row = visitors_at_min_hour.iloc[0]
        print(f"Visitor during the minimum morning hour:\n{user_row}")
    else:
        print("No visitors found during the minimum morning hour.")
else:
    print("No morning data available.")


# Для ночи - максимальный час

Visitor during the minimum morning hour:
datetime    2020-05-15 08:16:03.918402
year                              2020
month                                5
day                                 15
hour                                 8
minute                              16
second                               3
daytime                        morning
Name: alexander, dtype: object


In [42]:
night_rows = df[df['daytime'] == 'night']

if not night_rows.empty:
    max_hour_night = night_rows['hour'].max()

    visitors_at_max_hour = df[(df['daytime'] == 'night') & (df['hour'] == max_hour_night)]

    if not visitors_at_max_hour.empty:
        user_row = visitors_at_max_hour.iloc[0]
        print(f"Visitor during the maximum night hour:\n{user_row}")
    else:
        print("No visitors found during the maximum night hour.")
else:
    print("No night data available.")

Visitor during the maximum night hour:
datetime    2020-04-19 03:23:35.471598
year                              2020
month                                4
day                                 19
hour                                 3
minute                              23
second                              35
daytime                          night
Name: konstantin, dtype: object


In [43]:
# Посчитать модуль для столбца 'hour'
mode_hour = df['hour'].mode()[0]
print(f"Мода для часа:\n{mode_hour}")

Мода для часа:
22


In [44]:
# Посчитать модуль для столбца 'daytime'
mode_daytime = df['daytime'].mode()[0]
print(f"Мода для времени суток:\n{mode_daytime}")

Мода для времени суток:
evening


In [45]:
# Получение трех самых ранних часов и пользователей
earliest_three = df.nsmallest(3, 'hour')
print("Три самых ранних часа и пользователи:")
print(earliest_three.assign(user=earliest_three.index))

# Получение трех самых поздних часов и пользователей
latest_three = df.nlargest(3, 'hour')
print("Три самых поздних часа и пользователи:")
print(latest_three.assign(user=latest_three.index))

Три самых ранних часа и пользователи:
                            datetime  year  month  day  hour  minute  second  \
user                                                                           
valentina 2020-05-15 00:00:13.222265  2020      5   15     0       0      13   
valentina 2020-05-15 00:01:05.153738  2020      5   15     0       1       5   
pavel     2020-05-12 00:01:27.764025  2020      5   12     0       1      27   

          daytime       user  
user                          
valentina   night  valentina  
valentina   night  valentina  
pavel       night      pavel  
Три самых поздних часа и пользователи:
                            datetime  year  month  day  hour  minute  second  \
user                                                                           
ekaterina 2020-05-14 23:02:11.327532  2020      5   14    23       2      11   
ekaterina 2020-05-14 23:02:14.494985  2020      5   14    23       2      14   
ekaterina 2020-05-14 23:02:15.588808  2020     

In [46]:
# Получить базовую статистику по всему DataFrame
stats = df.describe()
print(stats)

# Предположим, что нас интересует столбец 'hour'
hour_stats = df['hour'].describe()
print(hour_stats)

# Вычислить интерквартильный размах (IQR)
iqr = hour_stats['75%'] - hour_stats['25%']
print(f"Интерквартильный размах (IQR) для часа: {iqr}")

                         datetime    year        month          day  \
count                        1076  1076.0  1076.000000  1076.000000   
mean   2020-05-10 09:00:41.211420  2020.0     4.870818    13.552974   
min    2020-04-17 12:01:08.463179  2020.0     4.000000     1.000000   
25%    2020-05-10 01:13:49.857472  2020.0     5.000000    11.000000   
50%    2020-05-11 22:48:35.302553  2020.0     5.000000    13.000000   
75%    2020-05-14 14:44:34.749530  2020.0     5.000000    15.000000   
max    2020-05-22 10:36:14.662600  2020.0     5.000000    30.000000   
std                           NaN     0.0     0.335557     4.906567   

              hour       minute       second  
count  1076.000000  1076.000000  1076.000000  
mean     16.249071    29.629182    29.500929  
min       0.000000     0.000000     0.000000  
25%      13.000000    14.000000    14.000000  
50%      19.000000    29.000000    30.000000  
75%      22.000000    46.000000    45.000000  
max      23.000000    59.000000

In [47]:
df.info()

<class 'pandas.DataFrame'>
Index: 1076 entries, valentina to alexander
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  1076 non-null   datetime64[us]
 1   year      1076 non-null   int32         
 2   month     1076 non-null   int32         
 3   day       1076 non-null   int32         
 4   hour      1076 non-null   int32         
 5   minute    1076 non-null   int32         
 6   second    1076 non-null   int32         
 7   daytime   1076 non-null   category      
dtypes: category(1), datetime64[us](1), int32(6)
memory usage: 75.6+ KB


In [48]:
df.count()

datetime    1076
year        1076
month       1076
day         1076
hour        1076
minute      1076
second      1076
daytime     1076
dtype: int64

In [49]:
df.daytime.value_counts()

daytime
evening          509
afternoon        252
early evening    145
night            129
morning           36
early morning      5
Name: count, dtype: int64

In [50]:
df.loc[df.daytime == 'night'].hour.idxmax()

'konstantin'

In [51]:
df.loc[df.daytime == 'morning'].hour.idxmin()

'alexander'

In [52]:
df.hour.mode()

0    22
Name: hour, dtype: int32

In [53]:
df.daytime.mode()

0    evening
Name: daytime, dtype: category
Categories (6, str): ['night' < 'early morning' < 'morning' < 'afternoon' < 'early evening' < 'evening']